# Топовый разделитель песен и восстановление аудио

Русский Gradio-интерфейс с BS-RoFormer, Mel-Band RoFormer, DeNoise и DeReverb.

Перед запуском выбери: **Среда выполнения → Сменить среду выполнения → T4 GPU**. Выполняй ячейки сверху вниз. Модели скачиваются только при первом использовании, а загруженная модель остаётся в памяти для повторной обработки, пока живёт сессия Colab.

In [ ]:
# 1. Проверка T4 и системная подготовка
import shutil
import subprocess

if not shutil.which("nvidia-smi"):
    raise RuntimeError("T4 GPU не найдена. Включи GPU в настройках среды выполнения Colab.")
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "ffmpeg", "git-lfs", "espeak-ng"],
    check=True,
)
subprocess.run(["git", "lfs", "install"], check=True)
if not shutil.which("uv") and not shutil.which("/root/.local/bin/uv"):
    subprocess.run(
        ["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
        check=True,
    )
print("T4 и системные программы готовы.")

In [ ]:
# 2. Получение проверяемой ветки и установка интерфейса
import os
import shutil
import subprocess
from pathlib import Path

repo = Path("/content/audio-restoration-colab")
repo_url = "https://github.com/egor125552/audio-restoration-colab.git"
branch = "agent/stem-separator-mixer"

if not (repo / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(repo)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-B", branch, f"origin/{branch}"], check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", f"origin/{branch}"], check=True)

uv = shutil.which("uv") or "/root/.local/bin/uv"
subprocess.run([uv, "python", "install", "3.11"], check=True)
subprocess.run(
    [uv, "venv", "--allow-existing", "--python", "3.11", str(repo / ".venv")],
    check=True,
)
subprocess.run(
    [uv, "pip", "install", "--python", str(repo / ".venv/bin/python"), str(repo)],
    check=True,
)
os.chdir(repo)
print("Интерфейс установлен из ветки", branch)

In [ ]:
# 3. Однократная подготовка движков разделения
import subprocess

subprocess.run(
    [
        "/content/audio-restoration-colab/scripts/prepare_backend.sh",
        "separator",
        "/content/audio-restoration-models",
    ],
    check=True,
)
print("Движки готовы. Веса выбранной модели скачаются при первом запуске и останутся в кэше.")

In [ ]:
# 4. Запуск. Открой появившуюся временную ссылку Gradio.
import os
import re
import subprocess
import threading
import time
from pathlib import Path

log_path = Path("/content/audio-restoration-gradio.log")
old_process = globals().get("gradio_process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    old_process.wait(timeout=15)
log_path.write_text("", encoding="utf-8")
gradio_process = subprocess.Popen(
    [
        "/content/audio-restoration-colab/.venv/bin/python",
        "-m",
        "audio_restoration_colab.app",
        "--share",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

def relay_gradio_output(process, path):
    with path.open("a", encoding="utf-8") as log_file:
        if process.stdout is None:
            return
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

gradio_log_thread = threading.Thread(
    target=relay_gradio_output,
    args=(gradio_process, log_path),
    daemon=True,
)
gradio_log_thread.start()
print("Запускаю интерфейс…", flush=True)
for _ in range(180):
    time.sleep(1)
    log_text = log_path.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"https://[^\s]+\.gradio\.live", log_text)
    if match:
        print("Интерфейс готов:", match.group(0))
        print("Общий лог:", log_path)
        break
    if gradio_process.poll() is not None:
        raise RuntimeError("Gradio завершился с ошибкой:\n" + log_text[-5000:])
else:
    gradio_process.terminate()
    raise RuntimeError("Gradio не выдал ссылку за 3 минуты:\n" + log_text[-5000:])